In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
safetensors_path = "/content/drive/MyDrive/model.safetensors"

先在 Colab 裝 transformers(!pip install -U transformers accelerate),跑 extract_real_activation.py。它會:
載入真實的 google/gemma-4-E2B-it-qat-mobile-transformers
跑一句真實句子(你可以改 PROMPT)
在 layer 15 的 mlp 模組前後掛 hook,抓「真正進去的 hidden state」跟「真正的 mlp 輸出」
存成三個 .npy 檔,其中 real_x_int_layer15.npy 已經幫你用 manifest 的 S_x 量化好了

In [3]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
extract_real_activation.py
===========================
從真實的 google/gemma-4-E2B-it-qat-mobile-transformers 模型跑一次
forward pass，在 layer 15 的 mlp 模組前後掛 hook，抓出：

  1) real_x_float_layer15.npy  : 真正進入 gate_proj/up_proj 的浮點 hidden
                                  state（也就是 pre_feedforward_layernorm
                                  之後、mlp() 呼叫之前的張量）
  2) real_x_int_layer15.npy    : 用 manifest.json 的 gate_proj_input scale
                                  (S_x) 量化成 int8 之後的版本，可以直接
                                  丟進 mlp_golden_layer15.py 當 x_int 用
  3) real_mlp_out_layer15.npy  : mlp() 這個 module 的真實輸出（down_proj
                                  的真實輸出，還沒加 residual、還沒過
                                  post_feedforward_layernorm）—— 這就是
                                  你的 golden model 應該要去逼近的「正確
                                  答案」

這一步是為了回答一個 mlp_golden_layer15.py 本身回答不了的問題：
「我的浮點參考模型（mlp_fp_reference），到底是不是 Gemma 真正在做的計算？」
如果連浮點版本都對不上真實模型的輸出，那量化誤差多小都沒有意義——問題根本
不在量化，而在演算法本身理解錯了。
"""

import json
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LAYER_IDX = 15
MODEL_ID = "google/gemma-4-E2B-it-qat-mobile-transformers"
MANIFEST_PATH = "/content/manifest.json"
PROMPT = "The quick brown fox jumps over the lazy dog."   # 換成你想測的真實句子

captured = {}


def pre_hook(module, args, kwargs):
    """抓 mlp.forward 的輸入（真實的 hidden state，量化前的浮點值）。"""
    if len(args) > 0:
        hs = args[0]
    else:
        hs = kwargs["hidden_states"]
    captured["x_float"] = hs.detach().to(torch.float64).cpu().numpy()


def post_hook(module, args, kwargs, output):
    """抓 mlp.forward 的輸出（down_proj 的真實輸出，還沒加 residual）。"""
    captured["mlp_out"] = output.detach().to(torch.float64).cpu().numpy()


def find_mlp_module(model, layer_idx):
    """不同模型的屬性路徑常常對不上文件範例，所以用 named_modules 搜尋
    名稱結尾是 'layers.{layer_idx}.mlp' 的模組。這個模型是多模態的，
    vision_tower / audio_tower 底下也有自己的 'layers.N.mlp'，所以要
    優先挑名字裡帶 'language_model' 的那個，其他一律排除。"""
    target_suffix = f"layers.{layer_idx}.mlp"
    all_candidates = [(name, mod) for name, mod in model.named_modules()
                      if name.endswith(target_suffix)]
    candidates = [(n, m) for n, m in all_candidates if "language_model" in n]

    if not candidates:
        print("找不到 language_model 底下符合的 mlp 模組，這是目前找到的"
              "所有候選（可能命名方式不一樣，自己核對）：")
        for name, _ in all_candidates:
            print("   ", name)
        if not all_candidates:
            for name, _ in model.named_modules():
                if "mlp" in name or f".{layer_idx}." in name:
                    print("   ", name)
        raise RuntimeError("adjust target_suffix / layer path and retry")
    if len(candidates) > 1:
        print(f"[warning] 找到 {len(candidates)} 個符合的模組，取第一個: "
              f"{candidates[0][0]}")
    else:
        print(f"[find_mlp_module] 鎖定模組: {candidates[0][0]}")
    return candidates[0][1]


def main():
    print(f"載入模型 {MODEL_ID} ...（第一次跑會下載，可能要一點時間）")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    # 全部用 float32：這個 checkpoint 內部有些地方（RMSNorm、QAT 的假量化
    # 模擬層）會刻意升到 float32 算完又轉回 bfloat16，如果那段轉型別沒做好
    # 就會跟其餘 bfloat16 權重對不上。全程 float32 可以避開這個混用問題，
    # 代價是吃的記憶體變大、跑起來變慢，2B 模型在 Colab 上通常還撐得住。
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, dtype=torch.float32, device_map="auto")
    model = model.to(torch.float32)
    model.eval()

    mlp_module = find_mlp_module(model, LAYER_IDX)
    print(f"掛 hook 在: layer {LAYER_IDX} 的 mlp 模組上")

    h1 = mlp_module.register_forward_pre_hook(pre_hook, with_kwargs=True)
    h2 = mlp_module.register_forward_hook(post_hook, with_kwargs=True)

    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
    with torch.no_grad():
        model(**inputs)

    h1.remove()
    h2.remove()

    if "x_float" not in captured or "mlp_out" not in captured:
        raise RuntimeError("hook 沒有抓到東西，先確認 mlp 模組路徑是否正確。")

    # hidden_states 形狀通常是 [batch, seq_len, hidden]，這裡先攤平成
    # [seq_len, hidden]，你可以自己選要用哪個 token 位置，或全部都存起來。
    x_float = captured["x_float"]
    mlp_out = captured["mlp_out"]
    print(f"real hidden state shape : {x_float.shape}")
    print(f"real mlp output shape   : {mlp_out.shape}")

    np.save("real_x_float_layer15.npy", x_float)
    np.save("real_mlp_out_layer15.npy", mlp_out)

    # ---- 用 manifest 的 gate_proj_input scale 量化成 int8，直接可以餵給
    #      mlp_golden_layer15.py 的 mlp_golden_int() ----
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    S_x = manifest["layers"][str(LAYER_IDX)]["activation_scales"]["gate_proj_input"]

    x_int = np.clip(np.round(x_float / S_x), -128, 127).astype(np.int64)
    np.save("real_x_int_layer15.npy", x_int)

    print(f"\n已存檔:")
    print(f"  real_x_float_layer15.npy  (真實浮點 hidden state)")
    print(f"  real_x_int_layer15.npy    (用 S_x={S_x:.6g} 量化後的 int8，"
          f"直接餵進你的 golden model)")
    print(f"  real_mlp_out_layer15.npy  (真實模型算出來的 mlp 輸出，用來當"
          f"最終比對的 ground truth，而不是你自己的 mlp_fp_reference)")
    print(f"\n輸入的 token 序列長度是 {x_float.shape[-2]}，如果你想先只測一個"
          f"token，可以在載入 .npy 後自己取某一個 [batch, position, :]。")


if __name__ == "__main__":
    main()

載入模型 google/gemma-4-E2B-it-qat-mobile-transformers ...（第一次跑會下載，可能要一點時間）


config.json:   0%|          | 0.00/6.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2550 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/209 [00:00<?, ?B/s]

[find_mlp_module] 鎖定模組: model.language_model.layers.15.mlp
掛 hook 在: layer 15 的 mlp 模組上
real hidden state shape : (1, 10, 1536)
real mlp output shape   : (1, 10, 1536)

已存檔:
  real_x_float_layer15.npy  (真實浮點 hidden state)
  real_x_int_layer15.npy    (用 S_x=0.023189 量化後的 int8，直接餵進你的 golden model)
  real_mlp_out_layer15.npy  (真實模型算出來的 mlp 輸出，用來當最終比對的 ground truth，而不是你自己的 mlp_fp_reference)

輸入的 token 序列長度是 10，如果你想先只測一個token，可以在載入 .npy 後自己取某一個 [batch, position, :]。


開始

In [6]:
!pip install safetensors

In [7]:
!unzip -o /content/layer_15.zip -d /content/layer_15

Archive:  /content/layer_15.zip
   creating: /content/layer_15/layer_15/
  inflating: /content/layer_15/layer_15/activation_scales.bin  
  inflating: /content/layer_15/layer_15/activation_scales.json  
  inflating: /content/layer_15/layer_15/down_scale_slr0.bin  
  inflating: /content/layer_15/layer_15/down_scale_slr1.bin  
  inflating: /content/layer_15/layer_15/down_scale_slr2.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc00.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc01.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc02.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc03.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc04.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc05.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc06.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc07.bin  
  inflating: /content/layer_15/layer_15/down_weight_slr0_pc08.bin  


In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
mlp_golden_layer15.py
======================
Gemma-4-E2B MLP golden model (Python) — bit-accurate translation of
mlp_model.h / gen_luts.py, generalized to read REAL weights + REAL
activation scales out of manifest.json instead of the synthetic wgen()
placeholder.

- 保留 check_v6 的 bin 檔讀取邏輯 (load_layer_bin)，可直接吃 INT2 / INT4 兩種。
- GELU 查表邏輯照抄 gen_luts.py 的 build_glut()，但 S_G / S_U / S_H 改成從
  manifest.json 讀真實值，不再用 placeholder。
- 捨入方式統一用 round-half-to-even (rne / rne_real)，跟 MLP.cpp 的
  rne_sh<> 位元對齊。
- 新增：down_proj 的最終輸出改成 INT16，用 manifest 的 down_proj_output
  scale 做這一次額外的量化，取代原本直接丟給 RMSNorm 的 Q7.8 / float 作法。
"""

import json
import math
import os
import numpy as np

# ===========================================================================
# 0. 位元精確的整數 helper（跟 gen_luts.py / mlp_model.h 完全一樣的邏輯）
# ===========================================================================

def rne(v: int, sh: int) -> int:
    """round-half-to-even of v * 2^-sh on Python ints (arbitrary precision)."""
    q = v >> sh
    rem = v - (q << sh)
    half = 1 << (sh - 1)
    if rem > half or (rem == half and (q & 1)):
        q += 1
    return q


def rne_real(x: float):
    """round-half-to-even of a real number."""
    f = math.floor(x)
    d = x - f
    if d > 0.5:
        return f + 1
    if d < 0.5:
        return f
    return f if (f % 2 == 0) else f + 1


def clamp8(v):
    return max(-128, min(127, v))


def clamp16(v):
    return max(-32768, min(32767, v))


K1 = 0.7978845608028654   # sqrt(2/pi)
K3 = 0.044715


def gelu_tanh(x):
    """gelu_pytorch_tanh — the SAME approximation gen_luts.py uses (matches
    Gemma). Do NOT swap this for the sigmoid quick-GELU used in the old
    check_v6 draft; that is a different function and will not match."""
    return 0.5 * x * (1.0 + np.tanh(K1 * (x + K3 * x ** 3)))


MLP_GLUT_FRAC = 13   # G' is Q3.13
MLP_RQ_SHIFT  = 20   # SRQ multipliers r_g/r_u: raw * 2^-20
MLP_CD_SHIFT  = 24   # c_down dequant scales:   raw * 2^-24


def build_glut(S_G: float, S_U: float, S_H: float, width_bits: int = 16):
    """Exact 256-entry G' table for THIS layer's real scales (not the
    placeholder 0.0625/0.0625/0.25 baked into gelu_luts.h).

    IMPORTANT: gelu_luts.h stores G' as `short` (int16, Q3.13 = 3 integer
    bits). That width was sized against the PLACEHOLDER ratio S_U/S_H=0.25.
    Real per-layer checkpoint scales can push S_U/S_H well above 1.0 (see
    manifest.json — it varies from ~0.02 to ~1.5 across layers), which makes
    some layers' G' entries not fit in int16 at all.

    width_bits=16 (default): mimic the CURRENT hardware exactly — saturate
        to the int16 range, and report how many entries / by how much. This
        is "what the chip actually computes today".
    width_bits=None: no artificial ceiling at all (only Python's arbitrary
        precision ints). Use this as a "what if we widened the table"
        reference to measure how much of the end-to-end error is caused
        purely by the int16 container, as opposed to genuine INT2/quantization
        noise that would exist either way.
    """
    g = np.empty(256, dtype=np.int64)
    n_over = 0
    max_abs_raw = 0
    lo, hi = (None, None)
    if width_bits is not None:
        hi = (1 << (width_bits - 1)) - 1
        lo = -(1 << (width_bits - 1))
    for i in range(256):
        qg = i - 128
        val = gelu_tanh(qg * S_G) * S_U / S_H * (1 << MLP_GLUT_FRAC)
        v = rne_real(val)
        max_abs_raw = max(max_abs_raw, abs(v))
        if width_bits is not None and not (lo <= v <= hi):
            n_over += 1
            v = max(lo, min(hi, v))
        g[i] = v
    if width_bits is not None and n_over:
        print(f"[build_glut] WARNING: {n_over}/256 G' entries overflow the "
              f"int{width_bits}/Q3.13 format (max |raw value| = {max_abs_raw}, "
              f"format max = {hi}) and were saturated. S_U/S_H = "
              f"{S_U / S_H:.4f} for this layer — the current "
              f"`short GELU_LUT_Q313[256]` format in gelu_luts.h was only "
              f"sized for the placeholder ratio 0.25. Flag this to your "
              f"HLS teammate: either widen the table to int32, or the "
              f"Q3.13 format needs to be revisited per-layer.")
    return g


# ===========================================================================
# 1. bin 檔讀取（沿用 check_v6，generic over num_bits — 2 或 4 都吃）
# ===========================================================================

def decode_already_flipped(data_u8, num_bits):
    factor = 8 // num_bits
    mask = (1 << num_bits) - 1
    sign_bit = 1 << (num_bits - 1)
    out = np.empty(data_u8.shape[:-1] + (data_u8.shape[-1] * factor,), dtype=np.int8)
    for i in range(factor):
        raw = (data_u8.astype(np.int32) >> (i * num_bits)) & mask
        signed = np.where(raw >= sign_bit, raw - (1 << num_bits), raw).astype(np.int8)
        out[..., i::factor] = signed
    return out


def offset_binary_to_twos_complement_mask(num_bits):
    factor = 8 // num_bits
    mask = 0
    for i in range(factor):
        mask |= (1 << (i * num_bits + (num_bits - 1)))
    return mask


MAGIC = {nb: offset_binary_to_twos_complement_mask(nb) for nb in (2, 4)}


def decode_offset_binary(data_u8, num_bits):
    """官方 safetensors 權重（還沒被同學的打包腳本 XOR 過 magic）的正確
    解碼方式：offset-binary，value = code - 2^(num_bits-1)。跟
    bin_packing_mlp.ipynb 裡的 unpack_to_int() 邏輯一致。這跟
    decode_already_flipped() 的 two's-complement 判斷不是同一套規則，
    不能混用——只有『已經被 XOR 過 magic』的位元組才能用
    decode_already_flipped() 解。"""
    factor = 8 // num_bits
    offset = 1 << (num_bits - 1)
    mask = (1 << num_bits) - 1
    out = np.empty(data_u8.shape[:-1] + (data_u8.shape[-1] * factor,), dtype=np.int8)
    for i in range(factor):
        sub = ((data_u8.astype(np.int32) >> (i * num_bits)) & mask).astype(np.int8) - offset
        out[..., i::factor] = sub
    return out


def load_layer_bin(layer_dir, num_bits, F, hidden_size=1536, NPC=32, DN=512):
    """回傳 W_gate, W_up, W_down_slrs(list of 3), S_gate, S_up, S_down.
    (act_scales 這次改成直接從 manifest.json 讀，見 main()，這個函式
    不再需要 activation_scales.json。)"""
    factor = 8 // num_bits
    BPC = hidden_size // factor
    N = 2 * F
    CPP = DN // NPC

    hbm_streams = []
    for p in range(NPC):
        path = os.path.join(layer_dir, f"gate_up_weight_pc{p:02d}.bin")
        hbm_streams.append(np.fromfile(path, dtype=np.uint8))

    n_tiles = N // DN
    fused_flipped = np.empty((N, BPC), dtype=np.uint8)
    for t in range(n_tiles):
        for p in range(NPC):
            offset = t * CPP * BPC
            seg = hbm_streams[p][offset: offset + CPP * BPC].reshape(CPP, BPC)
            fused_flipped[t * DN + p * CPP: t * DN + (p + 1) * CPP] = seg

    # 修正：同學的打包程式碼（bin_packing_mlp.ipynb）在寫檔前，已經把官方
    # safetensors 的 offset-binary 位元組 XOR 過 magic、轉成 two's-complement
    # 形式了。bin 檔裡存的就是「轉換後」的位元組，讀回來時應該直接用
    # decode_already_flipped() 解，不需要再 XOR 一次——之前這裡多做的
    # `^ magic` 等於把轉換又轉了回去，導致整段權重解碼系統性算錯。
    fused_int = decode_already_flipped(fused_flipped, num_bits)

    W_gate = fused_int[0::2, :]
    W_up = fused_int[1::2, :]

    scale_streams = []
    for p in range(NPC):
        path = os.path.join(layer_dir, f"gate_up_scale_pc{p:02d}.bin")
        scale_streams.append(np.fromfile(path, dtype=np.float32))

    combined_scale = np.empty((N,), dtype=np.float32)
    for t in range(n_tiles):
        for p in range(NPC):
            offset = t * CPP
            seg = scale_streams[p][offset: offset + CPP]
            combined_scale[t * DN + p * CPP: t * DN + (p + 1) * CPP] = seg

    S_gate = combined_scale[0::2]
    S_up = combined_scale[1::2]

    K_down = F
    BPC_FULL = K_down // factor
    SEG = BPC_FULL // 3
    n_lane = 128
    CPP_D = n_lane // NPC
    n_tiles_down = hidden_size // DN
    n_passes = DN // n_lane

    W_down_slrs = []
    for s in range(3):
        pc_streams = []
        for p in range(NPC):
            path = os.path.join(layer_dir, f"down_weight_slr{s}_pc{p:02d}.bin")
            pc_streams.append(np.fromfile(path, dtype=np.uint8))

        seg_flipped = np.empty((hidden_size, SEG), dtype=np.uint8)
        for t in range(n_tiles_down):
            for q in range(n_passes):
                for p in range(NPC):
                    offset = (t * n_passes + q) * CPP_D * SEG
                    blk = pc_streams[p][offset: offset + CPP_D * SEG].reshape(CPP_D, SEG)
                    row_base = t * DN + q * n_lane + p * CPP_D
                    seg_flipped[row_base: row_base + CPP_D] = blk

        # 同上：down_proj 的 bin 檔也已經是轉換後的位元組，直接解，不要再 XOR。
        seg_int = decode_already_flipped(seg_flipped, num_bits)
        W_down_slrs.append(seg_int)

    S_down = np.fromfile(os.path.join(layer_dir, "down_scale_slr0.bin"), dtype=np.float32)

    return W_gate, W_up, W_down_slrs, S_gate, S_up, S_down


# ===========================================================================
# 1b. 交叉驗證：bin 檔讀出來的權重/scale 是否跟 safetensors 原始權重一致
#     （這一步只是「確認讀檔沒讀錯」，跟後面的量化誤差分析是兩件事，
#      建議每次換層/換 bin 檔都先跑一次這個，通過了再往下看誤差）
# ===========================================================================

def verify_loaded_data(layer_dir, layer_idx, safetensors_path, num_bits, F,
                        hidden_size=1536):
    """把 load_layer_bin() 讀出來的 W_gate/W_up/W_down/S_gate/S_up/S_down
    跟 safetensors 裡的官方原始權重逐一比對。

    官方權重(還沒被同學的打包腳本 XOR 過 magic)要用 offset-binary
    解碼（decode_offset_binary，等同同學 bin_packing_mlp.ipynb 裡的
    unpack_to_int()），不能用 decode_already_flipped() 的 two's-complement
    判斷式去解——這兩種解碼方式在 code=2,3 這種情況會解出不同的值，混用
    會讓這個驗證「看起來過了」但其實兩邊都算錯（這正是之前發生的事）。

    回傳 True 代表 gate / up / down 三組權重與 scale 全部一致。
    """
    try:
        from safetensors import safe_open
    except ImportError:
        print("[verify_loaded_data] 沒有安裝 safetensors，先執行 "
              "`!pip install safetensors` 再重跑這個檢查。")
        return False

    W_gate, W_up, W_down_slrs, S_gate, S_up, S_down = load_layer_bin(
        layer_dir, num_bits, F, hidden_size=hidden_size)
    W_down = np.concatenate(W_down_slrs, axis=1)   # 3 個 SLR 沿 K 軸拼回 [hidden_size, F]

    prefix = f"model.language_model.layers.{layer_idx}.mlp"
    all_ok = True

    with safe_open(safetensors_path, framework="numpy") as f:
        keys = set(f.keys())
        checks = [
            ("gate_proj", W_gate, S_gate),
            ("up_proj",   W_up,   S_up),
            ("down_proj", W_down, S_down),
        ]
        for name, W_bin, S_bin in checks:
            weight_key = f"{prefix}.{name}.weight"
            scale_key = f"{prefix}.{name}.weight_scale"

            if weight_key not in keys:
                print(f"[verify_loaded_data] 找不到 {weight_key}，"
                      f"略過 {name} 的權重比對（可能 key 名稱不同，自己在"
                      f"safetensors 裡確認一下實際的 key）。")
                all_ok = False
                continue

            official_u8 = f.get_tensor(weight_key)
            official_int = decode_offset_binary(official_u8, num_bits)
            match_weight = np.array_equal(W_bin, official_int)

            match_scale = True
            if scale_key in keys:
                official_scale = f.get_tensor(scale_key).flatten()
                match_scale = np.allclose(S_bin, official_scale)
            else:
                print(f"[verify_loaded_data] 找不到 {scale_key}，"
                      f"scale 比對略過。")

            status = "通過" if (match_weight and match_scale) else "不一致"
            print(f"[verify_loaded_data] layer {layer_idx:>2} {name:10s} "
                  f"weight={'match' if match_weight else 'MISMATCH'}  "
                  f"scale={'match' if match_scale else 'MISMATCH'}  -> {status}")

            if not match_weight:
                diff = np.sum(W_bin != official_int)
                print(f"    不一致元素數: {diff} / {W_bin.size}")

            all_ok = all_ok and match_weight and match_scale

    return all_ok


# ===========================================================================
# 2. 硬體位元精確 golden model：GEMV -> SRQ -> G' LUT -> GEMV -> INT16
# ===========================================================================

def mlp_golden_int(x_int, W_gate, W_up, W_down_slrs, S_gate, S_up, S_down,
                    S_x, S_G, S_U, S_H, S_down_out, glut, verbose=False):
    """整數運算，位元對齊 MLP.cpp / mlp_model.h 的資料路徑。
    S_x     : gate_proj_input == up_proj_input（同一個殘差流輸入）
    S_G,S_U : gate_proj_output, up_proj_output（per-tensor，已折入 glut）
    S_H     : down_proj_input（per-tensor，folded scale，跟 glut 用同一個）
    S_down_out : down_proj_output —— 這次新加的、給 INT16 輸出用的 scale
    verbose : 印出每一段中間值的統計資訊，方便定位訊號在哪一步塌陷
    """
    F = W_gate.shape[0]
    N_DOWN = W_down_slrs[0].shape[0]

    # ---- gate / up projection：INT32 累加 ----
    acc_g = (W_gate.astype(np.int64) @ x_int.astype(np.int64))
    acc_u = (W_up.astype(np.int64) @ x_int.astype(np.int64))

    # ---- SRQ：per-channel effective scale 折成整數 multiplier r_g/r_u ----
    #      r[i] = round(S_eff[i] * 2^RQ_SHIFT)，跟 mlp_model.h 的 r_g/r_u 同義
    r_g = np.round((S_x * S_gate.astype(np.float64) / S_G) * (1 << MLP_RQ_SHIFT)).astype(np.int64)
    r_u = np.round((S_x * S_up.astype(np.float64) / S_U) * (1 << MLP_RQ_SHIFT)).astype(np.int64)

    q_g = np.empty(F, dtype=np.int64)
    q_u = np.empty(F, dtype=np.int64)
    for i in range(F):
        q_g[i] = clamp8(rne(int(acc_g[i]) * int(r_g[i]), MLP_RQ_SHIFT))
        q_u[i] = clamp8(rne(int(acc_u[i]) * int(r_u[i]), MLP_RQ_SHIFT))

    if verbose:
        print(f"[verbose] acc_g: min={acc_g.min()} max={acc_g.max()} "
              f"mean_abs={np.mean(np.abs(acc_g)):.1f}")
        print(f"[verbose] r_g:   min={r_g.min()} max={r_g.max()}")
        print(f"[verbose] q_g:   min={q_g.min()} max={q_g.max()} "
              f"非零比例={np.mean(q_g != 0):.4f}")
        print(f"[verbose] q_u:   min={q_u.min()} max={q_u.max()} "
              f"非零比例={np.mean(q_u != 0):.4f}")

    # ---- 精確 G' 查表 + 一次 rne 位移，取得 h（已經在 S_H 這個 scale 上）----
    h = np.empty(F, dtype=np.int64)
    for i in range(F):
        gp = int(glut[q_g[i] + 128])
        h[i] = clamp8(rne(gp * int(q_u[i]), MLP_GLUT_FRAC))

    if verbose:
        print(f"[verbose] h:     min={h.min()} max={h.max()} "
              f"非零比例={np.mean(h != 0):.4f}")

    # ---- down_proj：K 依三個 SLR 切開，各自 MAC 再加總（跨 SLR 整數加）----
    SEG_elements = W_down_slrs[0].shape[1]
    ps = np.zeros(N_DOWN, dtype=np.int64)
    for s in range(3):
        x_slice = h[s * SEG_elements:(s + 1) * SEG_elements]
        ps += W_down_slrs[s].astype(np.int64) @ x_slice.astype(np.int64)

    if verbose:
        print(f"[verbose] ps (down psum): min={ps.min()} max={ps.max()} "
              f"mean_abs={np.mean(np.abs(ps)):.1f}  非零比例={np.mean(ps != 0):.4f}")

    # ---- 唯一的一次 dequant：c_down[j] = round(S_H * S_down[j] * 2^CD_SHIFT) ----
    c_down = np.round(S_H * S_down.astype(np.float64) * (1 << MLP_CD_SHIFT)).astype(np.int64)
    # 修正：ypre 是 down_proj 的「真實浮點值」（例如 0.078 這種小數），不是
    # 一個該被捨入成整數編碼的量化 code。之前這裡誤用了 rne()（那是給
    # q_g/q_u/h 這種本來就該是整數的地方用的），把 0.078 這種小數直接捨去
    # 成整數 0，導致真實訊號（量級本來就小於 0.5）幾乎全部塌陷成 0。這裡
    # 只需要單純的浮點除法，不需要、也不應該做任何捨入。
    ypre = (ps.astype(np.float64) * c_down.astype(np.float64)) / float(1 << MLP_CD_SHIFT)

    # ---- 新增：量化成 INT16，接去 RMSNorm ----
    y_int16 = np.empty(N_DOWN, dtype=np.int64)
    for j in range(N_DOWN):
        y_int16[j] = clamp16(rne_real(ypre[j] / S_down_out))

    return h, ps, ypre, y_int16


# ===========================================================================
# 3. 全浮點參考模型（no quantization anywhere）— 用來量誤差的基準
# ===========================================================================

def gen_synthetic_x_int(hidden_size, std=24.0, seed=42):
    """比較貼近真實 hidden-state 分布的測試輸入：Gaussian，而不是均勻拉滿
    整個 int8 範圍 ([-128,128) uniform 等於每個維度都刻意拉到最大振幅，
    是不寫實的最壞情境)。

    std 目前是「先驗猜測」，抓在敏感度掃描顯示誤差已經穩定收斂的 16~32
    這個量級，不是校正過的真實數字。等你們拿到真實 dump 出來的 hidden
    state，第一件事就是把這個函式整個換掉，直接讀真實資料，而不是繼續
    調這個函式的參數。
    """
    rng = np.random.default_rng(seed)
    x = rng.normal(loc=0.0, scale=std, size=hidden_size)
    return np.clip(np.round(x), -128, 127).astype(np.int64)


def mlp_fp_reference(x_int, S_x, W_gate, W_up, W_down_slrs, S_gate, S_up, S_down):
    x_float = x_int.astype(np.float64) * S_x
    W_gate_f = W_gate.astype(np.float64) * S_gate.astype(np.float64)[:, None]
    W_up_f = W_up.astype(np.float64) * S_up.astype(np.float64)[:, None]
    W_down_f = np.concatenate(W_down_slrs, axis=1).astype(np.float64) * S_down.astype(np.float64)[:, None]

    gate_out = W_gate_f @ x_float
    up_out = W_up_f @ x_float
    act = gelu_tanh(gate_out) * up_out          # 用真正的 gelu_pytorch_tanh
    down_out = W_down_f @ act
    return down_out


# ===========================================================================
# 4. 誤差指標
# ===========================================================================

def evaluate(fp_output, quant_output, label=""):
    dot = np.dot(fp_output, quant_output)
    n1 = np.linalg.norm(fp_output)
    n2 = np.linalg.norm(quant_output)
    cos_sim = dot / (n1 * n2 + 1e-12)
    mae = np.mean(np.abs(fp_output - quant_output))
    rmse = np.sqrt(np.mean((fp_output - quant_output) ** 2))
    rel_rmse = rmse / (n1 / np.sqrt(len(fp_output)) + 1e-12)

    title = f"golden-model 對齊結果 {label}".strip()
    print(f"\n========= {title} =========")
    print(f"Cosine Similarity : {cos_sim:.6f}")
    print(f"MAE               : {mae:.6f}")
    print(f"Relative RMSE     : {rel_rmse:.6f}")
    return cos_sim, mae, rel_rmse


# ===========================================================================
# 5. main
# ===========================================================================

def main():
    LAYER_IDX = 15
    LAYER_DIR = "/content/layer_15/layer_15"     # 依實際解壓縮路徑調整
    MANIFEST_PATH = "/content/manifest.json"
    # SAFETENSORS_PATH = "/content/model.safetensors"   # 依實際上傳路徑調整

    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    layer_cfg = manifest["layers"][str(LAYER_IDX)]

    num_bits = layer_cfg["num_bits"]              # 15 層 -> 2 (INT2)
    F = layer_cfg["F"]                             # 12288
    hidden_size = manifest["hidden_size"]           # 1536
    scales = layer_cfg["activation_scales"]

    S_x       = scales["gate_proj_input"]           # == up_proj_input
    S_G       = scales["gate_proj_output"]
    S_U       = scales["up_proj_output"]
    S_H       = scales["down_proj_input"]
    S_down_out = scales["down_proj_output"]          # 新增：INT16 輸出用

    assert scales["gate_proj_input"] == scales["up_proj_input"], \
        "gate/up 的輸入 scale 理論上應該相同（同一個殘差流輸入）"

    print(f"[layer {LAYER_IDX}] num_bits={num_bits}  F={F}")
    print(f"  S_x={S_x:.6g}  S_G={S_G:.6g}  S_U={S_U:.6g}  "
          f"S_H={S_H:.6g}  S_down_out={S_down_out:.6g}")

    # ---- Step 0：先驗證 bin 檔讀取是否跟 safetensors 官方權重一致 ----
    print("\n--- Step 0: 驗證 bin 檔讀取是否跟 safetensors 一致 ---")
    verify_ok = verify_loaded_data(
        LAYER_DIR, LAYER_IDX, safetensors_path, num_bits, F,
        hidden_size=hidden_size)
    if not verify_ok:
        print("[main] WARNING: bin 檔驗證沒有全部通過——下面 golden model "
              "算出來的誤差可能混進了「讀檔本身讀錯」的雜訊，建議先把這裡"
              "的不一致排除掉，再回頭看量化誤差的數字才有意義。\n")
    else:
        print("[main] bin 檔驗證通過，繼續往下跑 golden model。\n")

    # ---- 讀 bin 檔（golden model 用；上面驗證時其實已經讀過一次，這裡為了
    #      程式邏輯單純分開寫，之後想省時間可以讓 verify_loaded_data 直接
    #      回傳這幾個陣列重複使用）----
    W_gate, W_up, W_down_slrs, S_gate, S_up, S_down = load_layer_bin(
        LAYER_DIR, num_bits, F, hidden_size=hidden_size)

    # ---- 用這一層真實的 S_G/S_U/S_H 重建 G' 表 ----
    #      current  : 跟現在 gelu_luts.h 的 int16/Q3.13 格式一模一樣 (會飽和)
    #      wide_ref : 表格寬度無上限，只是拿來量「格式限制本身」造成多少誤差
    glut_current = build_glut(S_G, S_U, S_H, width_bits=16)
    glut_wide    = build_glut(S_G, S_U, S_H, width_bits=None)

    # ---- 測試輸入：優先用 extract_real_activation.py 產生的真實資料，
    #      沒有的話 fallback 成合成 Gaussian（std=24 是先驗猜測，見敏感度
    #      掃描）。real_mlp_out 有的話，會拿真實模型的輸出當 ground truth，
    #      而不是只跟自己的 mlp_fp_reference 比。----
    import os as _os
    real_x_path = "real_x_int_layer15.npy"
    real_out_path = "real_mlp_out_layer15.npy"
    real_target = None

    if _os.path.exists(real_x_path):
        x_int_full = np.load(real_x_path)          # 形狀可能是 [1, seq, hidden]
        x_int = x_int_full.reshape(-1, hidden_size)[-1]   # 先取最後一個 token
        print(f"[main] 偵測到真實 activation ({real_x_path})，改用真實輸入 "
              f"(取序列最後一個 token)。")
        if _os.path.exists(real_out_path):
            real_out_full = np.load(real_out_path)
            real_target = real_out_full.reshape(-1, hidden_size)[-1].astype(np.float64)
            print(f"[main] 偵測到真實 mlp 輸出 ({real_out_path})，會拿它當"
                  f"最終 ground truth，而不是只用 mlp_fp_reference。")
    else:
        x_int = gen_synthetic_x_int(hidden_size, std=24.0, seed=42)
        print(f"[main] 沒有找到 {real_x_path}，先用合成 Gaussian 輸入 "
              f"(std=24)。想驗證真實誤差，先跑 extract_real_activation.py。")

    # ---- 跑兩組硬體位元精確 golden model ----
    print("\n---- [verbose] 中間值統計（找訊號在哪一步塌陷用）----")
    h_cur,  ps_cur,  ypre_cur,  y16_cur = mlp_golden_int(
        x_int, W_gate, W_up, W_down_slrs, S_gate, S_up, S_down,
        S_x, S_G, S_U, S_H, S_down_out, glut_current, verbose=True)
    h_wide, ps_wide, ypre_wide, y16_wide = mlp_golden_int(
        x_int, W_gate, W_up, W_down_slrs, S_gate, S_up, S_down,
        S_x, S_G, S_U, S_H, S_down_out, glut_wide)

    # ---- 跑全浮點參考 ----
    fp_out = mlp_fp_reference(x_int, S_x, W_gate, W_up, W_down_slrs, S_gate, S_up, S_down)

    y16_cur_dq  = y16_cur.astype(np.float64)  * S_down_out
    y16_wide_dq = y16_wide.astype(np.float64) * S_down_out

    def clamp_hits(h):
        return int((h == 127).sum() + (h == -128).sum())

    print(f"\n[現況 int16 格式]  h clamp 命中數 = {clamp_hits(h_cur)}  / {len(h_cur)}"
          f"   ({100*clamp_hits(h_cur)/len(h_cur):.1f}%)")
    print(f"[理想寬表格]       h clamp 命中數 = {clamp_hits(h_wide)} / {len(h_wide)}"
          f"   ({100*clamp_hits(h_wide)/len(h_wide):.1f}%)")
    print("  -> 兩者的差，就是「純粹因為 int16 表格塞不下」多出來的 clamp，"
          "不是 INT2 權重本身該有的誤差。")

    print(f"\nFP 參考前5筆         : {fp_out[:5]}")
    print(f"[現況] INT16 dequant : {y16_cur_dq[:5]}")
    print(f"[理想] INT16 dequant : {y16_wide_dq[:5]}")

    print("\n---- [現況 int16 格式] ----")
    evaluate(fp_out, y16_cur_dq, label="[現況 int16 格式] vs 自己的浮點參考")
    print("\n---- [理想寬表格，用來量格式限制本身的影響] ----")
    evaluate(fp_out, y16_wide_dq, label="[理想寬表格] vs 自己的浮點參考")

    if real_target is not None:
        # ---- 這一段才是真正回答「我的演算法有沒有偏離原本模型」這件事：
        #      拿真實模型算出來的 mlp 輸出當 ground truth，分別跟(a) 你自己
        #      推導的浮點參考、(b) 你的整數 golden model 比對。如果 (a) 就
        #      已經對不太上，代表問題出在演算法本身理解錯了，不是量化。----
        print("\n==================== 對照真實模型輸出 ====================")
        evaluate(real_target, fp_out, label="[自己的浮點參考] vs [真實模型輸出]")
        evaluate(real_target, y16_cur_dq, label="[現況 int16 golden model] vs [真實模型輸出]")

    # ------------------------------------------------------------------
    # 診斷：目前的隨機測試向量是「每個維度都獨立均勻分布在整個 int8 範圍」，
    # 這是刻意的最壞情境輸入，不代表真實 hidden state 的分布。掃過幾種
    # 「輸入音量」(振幅) 看 h 的 clamp 比例與誤差怎麼變，可以幫你判斷：現在
    # 看到的大誤差，有多少其實是「測試輸入太極端」造成的假象。
    # 等你們有真實 dump 出來的 hidden state，直接把那個丟進來取代 x_int，
    # 才是真正有意義的誤差數字。
    # ------------------------------------------------------------------
    print("\n---- 輸入標準差敏感度掃描 (仍用 glut_current，Gaussian 合成輸入) ----")
    print(f"{'std':>10} {'h clamp %':>10} {'cos_sim':>10} {'rel_RMSE':>10}")
    for std in (64.0, 48.0, 32.0, 24.0, 16.0, 8.0):
        x_test = gen_synthetic_x_int(hidden_size, std=std, seed=42)
        h_t, ps_t, ypre_t, y16_t = mlp_golden_int(
            x_test, W_gate, W_up, W_down_slrs, S_gate, S_up, S_down,
            S_x, S_G, S_U, S_H, S_down_out, glut_current)
        fp_t = mlp_fp_reference(x_test, S_x, W_gate, W_up, W_down_slrs, S_gate, S_up, S_down)
        y16_t_dq = y16_t.astype(np.float64) * S_down_out
        clamp_pct = 100 * clamp_hits(h_t) / len(h_t)
        cos_t = np.dot(fp_t, y16_t_dq) / (np.linalg.norm(fp_t) * np.linalg.norm(y16_t_dq) + 1e-12)
        rmse_t = np.sqrt(np.mean((fp_t - y16_t_dq) ** 2))
        rel_rmse_t = rmse_t / (np.linalg.norm(fp_t) / np.sqrt(len(fp_t)) + 1e-12)
        print(f"{std:>10.1f} {clamp_pct:>9.1f}% {cos_t:>10.4f} {rel_rmse_t:>10.4f}")
    print("如果 relative RMSE 隨 std 變小而大幅下降，代表現在看到的誤差主要是")
    print("「測試輸入太極端」造成的假象，不是硬體設計本身的問題——這時候最")
    print("優先要做的事，是換成真實 dump 出來的 hidden state 再重跑一次。")


if __name__ == "__main__":
    main()

[layer 15] num_bits=2  F=12288
  S_x=0.023189  S_G=0.0305118  S_U=0.0305118  S_H=0.0200541  S_down_out=0.00810574

--- Step 0: 驗證 bin 檔讀取是否跟 safetensors 一致 ---
[verify_loaded_data] layer 15 gate_proj  weight=match  scale=match  -> 通過
[verify_loaded_data] layer 15 up_proj    weight=match  scale=match  -> 通過
[verify_loaded_data] layer 15 down_proj  weight=match  scale=match  -> 通過
[main] bin 檔驗證通過，繼續往下跑 golden model。

[build_glut] WARNING: 41/256 G' entries overflow the int16/Q3.13 format (max |raw value| = 48296, format max = 32767) and were saturated. S_U/S_H = 1.5215 for this layer — the current `short GELU_LUT_Q313[256]` format in gelu_luts.h was only sized for the placeholder ratio 0.25. Flag this to your HLS teammate: either widen the table to int32, or the Q3.13 format needs to be revisited per-layer.
[main] 偵測到真實 activation (real_x_int_layer15.npy)，改用真實輸入 (取序列最後一個 token)。
[main] 偵測到真實 mlp 輸出 (real_mlp_out_layer15.npy)，會拿它當最終 ground truth，而不是只用 mlp_fp_reference。

---- [verbose] 中間

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
bisect_mlp_stages.py
======================
上一輪拿「整個 mlp() 的真實輸出」跟自己的浮點參考比對，cos_sim 只有
0.04，幾乎等於兩個不相關的向量。這代表問題不在量化精度，是浮點參考
本身就沒有算出真實模型在算的東西。

這支程式把 gate_proj / up_proj / down_proj 三個 Linear 分別掛 hook，
抓下「真實輸入」跟「真實輸出」，逐段跟「你自己解出來的 int 權重 *
scale」算出來的浮點值比對，藉此精確定位是從哪一段開始分岔：

  - gate_proj 真實輸出 vs (你的 W_gate 解碼結果 * S_gate) @ 真實輸入
  - up_proj   真實輸出 vs (你的 W_up   解碼結果 * S_up  ) @ 真實輸入
  - down_proj 真實輸出 vs (你的 W_down 解碼結果 * S_down) @ 真實 GELU*up

只要有一段的 cos_sim 掉到接近 0，問題就是那一段（通常是：weight 解碼
方式錯、activation function 選錯、或 GLU 的 gate/up 順序搞反）。
"""

import json
import os
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LAYER_IDX = 15
MODEL_ID = "google/gemma-4-E2B-it-qat-mobile-transformers"
MANIFEST_PATH = "/content/manifest.json"
LAYER_DIR = "/content/layer_15/layer_15"
PROMPT = "The quick brown fox jumps over the lazy dog."

captured = {}


def make_hook(name):
    def hook(module, args, kwargs, output):
        x = args[0] if len(args) > 0 else kwargs.get("input")
        captured[f"{name}_in"] = x.detach().to(torch.float64).cpu().numpy()
        captured[f"{name}_out"] = output.detach().to(torch.float64).cpu().numpy()
    return hook


def find_submodule(model, layer_idx, subname):
    target = f"layers.{layer_idx}.mlp.{subname}"
    for name, mod in model.named_modules():
        if name.endswith(target) and "language_model" in name:
            return name, mod
    raise RuntimeError(f"找不到 {target}（在 language_model 底下），"
                        f"自己用 model.named_modules() 核對實際路徑")


# ---------------------------------------------------------------------------
# 跟 mlp_golden_layer15.py 一樣的 bin 檔讀取邏輯（複製一份，保持這支診斷
# 工具可以獨立執行，不用依賴另一個檔案的 import 路徑）
# ---------------------------------------------------------------------------

def decode_already_flipped(data_u8, num_bits):
    factor = 8 // num_bits
    mask = (1 << num_bits) - 1
    sign_bit = 1 << (num_bits - 1)
    out = np.empty(data_u8.shape[:-1] + (data_u8.shape[-1] * factor,), dtype=np.int8)
    for i in range(factor):
        raw = (data_u8.astype(np.int32) >> (i * num_bits)) & mask
        signed = np.where(raw >= sign_bit, raw - (1 << num_bits), raw).astype(np.int8)
        out[..., i::factor] = signed
    return out


def offset_binary_to_twos_complement_mask(num_bits):
    factor = 8 // num_bits
    mask = 0
    for i in range(factor):
        mask |= (1 << (i * num_bits + (num_bits - 1)))
    return mask


MAGIC = {nb: offset_binary_to_twos_complement_mask(nb) for nb in (2, 4)}


def load_layer_bin(layer_dir, num_bits, F, hidden_size=1536, NPC=32, DN=512):
    factor = 8 // num_bits
    BPC = hidden_size // factor
    N = 2 * F
    CPP = DN // NPC

    hbm_streams = [np.fromfile(os.path.join(layer_dir, f"gate_up_weight_pc{p:02d}.bin"),
                                dtype=np.uint8) for p in range(NPC)]
    n_tiles = N // DN
    fused_flipped = np.empty((N, BPC), dtype=np.uint8)
    for t in range(n_tiles):
        for p in range(NPC):
            offset = t * CPP * BPC
            seg = hbm_streams[p][offset: offset + CPP * BPC].reshape(CPP, BPC)
            fused_flipped[t * DN + p * CPP: t * DN + (p + 1) * CPP] = seg
    # 修正：bin 檔裡存的位元組已經是同學打包時 XOR 過 magic 的結果，
    # 這裡不該再 XOR 一次，直接用 decode_already_flipped() 解。
    fused_int = decode_already_flipped(fused_flipped, num_bits)
    W_gate = fused_int[0::2, :]
    W_up = fused_int[1::2, :]

    scale_streams = [np.fromfile(os.path.join(layer_dir, f"gate_up_scale_pc{p:02d}.bin"),
                                  dtype=np.float32) for p in range(NPC)]
    combined_scale = np.empty((N,), dtype=np.float32)
    for t in range(n_tiles):
        for p in range(NPC):
            offset = t * CPP
            combined_scale[t * DN + p * CPP: t * DN + (p + 1) * CPP] = scale_streams[p][offset: offset + CPP]
    S_gate = combined_scale[0::2]
    S_up = combined_scale[1::2]

    K_down = F
    BPC_FULL = K_down // factor
    SEG = BPC_FULL // 3
    n_lane = 128
    CPP_D = n_lane // NPC
    n_tiles_down = hidden_size // DN
    n_passes = DN // n_lane

    W_down_slrs = []
    for s in range(3):
        pc_streams = [np.fromfile(os.path.join(layer_dir, f"down_weight_slr{s}_pc{p:02d}.bin"),
                                   dtype=np.uint8) for p in range(NPC)]
        seg_flipped = np.empty((hidden_size, SEG), dtype=np.uint8)
        for t in range(n_tiles_down):
            for q in range(n_passes):
                for p in range(NPC):
                    offset = (t * n_passes + q) * CPP_D * SEG
                    blk = pc_streams[p][offset: offset + CPP_D * SEG].reshape(CPP_D, SEG)
                    row_base = t * DN + q * n_lane + p * CPP_D
                    seg_flipped[row_base: row_base + CPP_D] = blk
        seg_int = decode_already_flipped(seg_flipped, num_bits)
        W_down_slrs.append(seg_int)

    S_down = np.fromfile(os.path.join(layer_dir, "down_scale_slr0.bin"), dtype=np.float32)
    return W_gate, W_up, W_down_slrs, S_gate, S_up, S_down


def cos_mae(a, b, label):
    cos = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)
    mae = np.mean(np.abs(a - b))
    print(f"{label:45s} cos_sim={cos: .6f}   MAE={mae:.6f}")
    return cos


K1 = 0.7978845608028654
K3 = 0.044715


def gelu_tanh(x):
    return 0.5 * x * (1.0 + np.tanh(K1 * (x + K3 * x ** 3)))


def main():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    layer_cfg = manifest["layers"][str(LAYER_IDX)]
    num_bits = layer_cfg["num_bits"]
    F = layer_cfg["F"]
    hidden_size = manifest["hidden_size"]
    scales = layer_cfg["activation_scales"]
    S_x = scales["gate_proj_input"]
    S_G = scales["gate_proj_output"]
    S_U = scales["up_proj_output"]
    S_H = scales["down_proj_input"]

    print(f"config.hidden_activation / hidden_act 的真實值待模型載入後印出\n")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, trust_remote_code=True, dtype=torch.float32, device_map="auto")
    model = model.to(torch.float32)
    model.eval()

    print("hidden_activation:", getattr(model.config, "hidden_activation", None))
    print("hidden_act       :", getattr(model.config, "hidden_act", None))
    # 如果是巢狀 text_config 也印出來看看
    if hasattr(model.config, "text_config"):
        print("text_config.hidden_activation:",
              getattr(model.config.text_config, "hidden_activation", None))
        print("text_config.hidden_act       :",
              getattr(model.config.text_config, "hidden_act", None))

    handles = []
    for sub in ("gate_proj", "up_proj", "down_proj"):
        name, mod = find_submodule(model, LAYER_IDX, sub)
        print(f"hook -> {name}")
        handles.append(mod.register_forward_hook(make_hook(sub), with_kwargs=True))

    inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
    with torch.no_grad():
        model(**inputs)
    for h in handles:
        h.remove()

    # 取序列最後一個 token 的位置，跟 mlp_golden_layer15.py 保持一致
    def last_tok(name):
        arr = captured[name]
        return arr.reshape(-1, arr.shape[-1])[-1].astype(np.float64)

    gate_in = last_tok("gate_proj_in")     # 真實進 gate_proj 的浮點輸入
    gate_out_real = last_tok("gate_proj_out")
    up_in = last_tok("up_proj_in")
    up_out_real = last_tok("up_proj_out")
    down_in_real = last_tok("down_proj_in")   # 真實進 down_proj 的浮點輸入
    down_out_real = last_tok("down_proj_out")

    print(f"\ngate_proj_in 跟 up_proj_in 是否相同輸入: "
          f"{np.allclose(gate_in, up_in)}")

    # ---- 讀你自己的 bin 檔權重，dequant 成浮點 ----
    W_gate, W_up, W_down_slrs, S_gate, S_up, S_down = load_layer_bin(
        LAYER_DIR, num_bits, F, hidden_size=hidden_size)
    W_gate_f = W_gate.astype(np.float64) * S_gate.astype(np.float64)[:, None]
    W_up_f = W_up.astype(np.float64) * S_up.astype(np.float64)[:, None]
    W_down_f = np.concatenate(W_down_slrs, axis=1).astype(np.float64) * S_down.astype(np.float64)[:, None]

    print("\n========== 逐段比對 ==========")
    my_gate_out = W_gate_f @ gate_in
    cos_mae(gate_out_real, my_gate_out, "gate_proj: 真實輸出 vs 你的 dequant 權重")

    my_up_out = W_up_f @ up_in
    cos_mae(up_out_real, my_up_out, "up_proj:   真實輸出 vs 你的 dequant 權重")

    # down_proj 的真實輸入，直接就是 GELU(gate)*up 的真實結果，不用自己重算
    my_act = gelu_tanh(gate_out_real) * up_out_real
    cos_mae(down_in_real, my_act, "GELU(真實gate)*真實up vs 真實down_proj輸入")

    my_down_out = W_down_f @ down_in_real
    cos_mae(down_out_real, my_down_out, "down_proj: 真實輸出 vs 你的 dequant 權重")

    # 額外：用「你自己算的 gate/up_out」接力算到底，跟真實 down_proj 輸出比
    my_act_from_my_gate_up = gelu_tanh(my_gate_out) * my_up_out
    my_down_out_e2e = W_down_f @ my_act_from_my_gate_up
    cos_mae(down_out_real, my_down_out_e2e, "down_proj: 真實輸出 vs 全程用你自己算的中間值")

    print("\n如果 gate_proj / up_proj 那兩行 cos_sim 就已經很低，問題出在權重")
    print("dequant（scale 套用方式、bit 解碼方式，或 gate/up 指派反了）。")
    print("如果 gate_proj/up_proj 準，但『GELU*up vs 真實down_proj輸入』那行不準，")
    print("問題出在 activation function 選錯，或 GLU 的組合方式不是 act(gate)*up。")
    print("如果前三行都準，只有最後 down_proj 那行不準，問題在 down_proj 的權重")
    print("dequant 或你的 SLR 切分/拼接順序。")


if __name__ == "__main__":
    main()